# U.S. Treasury 2025 Daily Par Yield Curve Quality Check

## tl;dr
This companion notebook profiles the retained Treasury source and its long-form normalization. It must not be interpreted as a trading-data validation or a point-in-time availability proof.

**Execution status:** not executed in this environment because Jupyter/nbformat/nbclient are not installed. The equivalent deterministic scripts have been run; see `../quality/2026-09-22_treasury_yield_curve_quality.md`.

## Context & Methods

- Raw source: `../raw/treasury_daily_par_yield_curve_2025_source.csv`
- Normalized source: `../processed/treasury_daily_par_yield_curve_2025_long_v1.csv`
- Grain: one `(observation_date, tenor)` row.
- Missing Treasury `N/A` values remain missing; no interpolation or zero fill is allowed.

In [ ]:
from pathlib import Path
import pandas as pd

datasets_dir = Path('research/datasets').resolve() if (Path('research/datasets').exists()) else Path('..').resolve()
raw_path = datasets_dir / 'raw' / 'treasury_daily_par_yield_curve_2025_source.csv'
normalized_path = datasets_dir / 'processed' / 'treasury_daily_par_yield_curve_2025_long_v1.csv'
raw = pd.read_csv(raw_path)
normalized = pd.read_csv(normalized_path)
raw.shape, normalized.shape

## Data

Inspect the input schema and bounded preview before relying on calculated checks.

In [ ]:
normalized.head(10)

## Results

Check expected rows, composite-key uniqueness, and source-held missingness.

In [ ]:
quality = {
    'raw_daily_rows': len(raw),
    'normalized_rows': len(normalized),
    'duplicate_date_tenor_keys': int(normalized.duplicated(['observation_date', 'tenor']).sum()),
    'source_missing_rows': int((normalized['quality_flag'] == 'source_missing_or_not_applicable').sum()),
    'missing_by_tenor': normalized.loc[normalized['quality_flag'] == 'source_missing_or_not_applicable', 'tenor'].value_counts().to_dict(),
}
quality

## Takeaways

- Use only after confirming the downstream legal/use boundary and historical availability timestamps.
- Treat Treasury `N/A` as missing source data, not as a zero rate or an interpolation target.
- Rerun `../scripts/validate_treasury_yield_curve_2025.py` after any source or transform change.